# Check: is CESM2-WACCM SSP2-4.5 (Pangeo CMIP6) daily `tasmax` identical to `tas`?
The pipeline's automatic check flagged it (difference 0.00 K), so CESM currently uses `tas` in all scenarios. This notebook compares the raw stores directly, for every SSP2-4.5 member, and also checks the Hub's G6 `TREFHTMX`.

In [1]:
import json, numpy as np, xarray as xr
cat = json.load(open("pyrosai_catalog.json"))
rows = []
for mem, e in sorted(cat["CESM2-WACCM"]["ssp245"].items()):
    if not ("tasmax" in e and "tas" in e and e["tasmax"].get("kind") == "zarr"):
        continue
    a = xr.open_zarr(e["tasmax"]["store"], storage_options={"anon": True})["tasmax"].isel(time=slice(0, 10)).load()
    b = xr.open_zarr(e["tas"]["store"], storage_options={"anon": True})["tas"].isel(time=slice(0, 10)).load()
    rows.append((mem, float(a.mean()), float(b.mean()), float(abs(a.values - b.values).max()),
                 a.attrs.get("cell_methods"), str(a.time.values[0])[:10], str(b.time.values[0])[:10]))
print("member      mean_tasmax  mean_tas  max|diff|  tasmax cell_methods                 t0_tasmax   t0_tas")
for r in rows:
    print(f"{r[0]:10s}  {r[1]:10.2f}  {r[2]:8.2f}  {r[3]:9.3f}  {str(r[4])[:35]:35s} {r[5]}  {r[6]}")

member      mean_tasmax  mean_tas  max|diff|  tasmax cell_methods                 t0_tasmax   t0_tas
r2i1p1f1        278.59    278.59      0.000  area: mean time: maximum            2015-01-01  2015-01-01
r3i1p1f1        278.28    278.28      0.000  area: mean time: maximum            2015-01-01  2015-01-01
r4i1p1f1        278.57    278.57      0.000  area: mean time: maximum            2015-01-01  2015-01-01
r5i1p1f1        278.24    278.24      0.000  area: mean time: maximum            2015-01-01  2015-01-01


In [2]:
# Same check on the Hub's CESM G6 daily output (TREFHTMX vs TREFHT, first file, first 10 days)
import sys, os; sys.path.insert(0, os.getcwd()); import pyrosai as P
c = P.apply_known_issues(cat)
for scen in ("G6-1.5K-SAI", "G6-1.5K-HiLLA"):
    e = c["CESM2-WACCM"][scen]["r1"]
    one = lambda v: xr.open_dataset(__import__("fsspec").open(e[v]["paths"][0]).open(), engine="h5netcdf")[e[v]["var"]].isel(time=slice(0, 10)).load()
    try:
        a, b = one("tasmax"), one("tas")
        print(scen, "mean tasmax %.2f  mean tas %.2f  max|diff| %.3f" % (float(a.mean()), float(b.mean()), float(abs(a.values - b.values).max())))
    except Exception as ex:
        print(scen, "could not read:", type(ex).__name__, ex)

[catalog] CESM2-WACCM: ignored 298 land/ocean/ice-model file(s) (e.g. CLM LDAY/LMON)


[known issue] MIROC-ES2H: removed 'sfcWind' from 6 member(s) - MIROC-ES2H 'sfcWind' files contain a radiative flux (units W/m**2, mean ~160-200), not wind speed (found 2026-10-01)


[wind] MIROC-ES2H: FWI wind = SSP2-4.5 day-of-year climatology of hypot(uas,vas) in ALL scenarios


G6-1.5K-SAI mean tasmax 280.16  mean tas 278.69  max|diff| 18.270


G6-1.5K-HiLLA mean tasmax 280.15  mean tas 278.69  max|diff| 18.326
